In [ ]:

from google.colab import userdata
import os
import subprocess
import base64

token = userdata.get("GH_TOKEN")
repo = "zeeshanamin332-dot/quant-ai-trading-platform"

# Pass credentials through Git's temporary configuration,
# not through the visible clone URL.
env = os.environ.copy()
env["GIT_CONFIG_COUNT"] = "1"
env["GIT_CONFIG_KEY_0"] = "http.https://github.com/.extraheader"
env["GIT_CONFIG_VALUE_0"] = (
    "AUTHORIZATION: basic "
    + base64.b64encode(
        f"x-access-token:{token}".encode()
    ).decode()
)

subprocess.run(
    ["git", "clone", f"https://github.com/{repo}.git"],
    env=env,
    check=True
)

print("Private repository cloned successfully.")


Private repository cloned successfully.


In [ ]:

from pathlib import Path

PROJECT = Path("/content/quant-ai-trading-platform")

folders = [
    "src/data",
    "src/strategies",
    "src/backtesting",
    "notebooks",
    "tests",
    "data",
    "results",
]

for folder in folders:
    (PROJECT / folder).mkdir(parents=True, exist_ok=True)

print("Project folders created.")


Project folders created.


In [ ]:

%pip -q install ccxt pandas numpy matplotlib


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 157.4/157.4 kB 6.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.7/6.7 MB 83.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 219.5/219.5 kB 22.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 215.6/215.6 kB 21.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 72.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.5/65.5 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.5/61.5 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.7/135.7 kB 15.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.4/4.4 MB 97.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.4/132.4 kB 15.5 MB/s eta 0:00:00


In [ ]:

import ccxt
import pandas as pd

exchange = ccxt.kraken({"enableRateLimit": True})

symbol = "BTC/USD"
timeframe = "1h"
limit = 500

candles = exchange.fetch_ohlcv(
    symbol,
    timeframe=timeframe,
    limit=limit
)

df = pd.DataFrame(
    candles,
    columns=[
        "timestamp", "open", "high",
        "low", "close", "volume"
    ]
)

df["timestamp"] = pd.to_datetime(
    df["timestamp"], unit="ms", utc=True
)

df = (
    df.drop_duplicates("timestamp")
      .sort_values("timestamp")
      .set_index("timestamp")
)

df = df.astype(float)

assert df.index.is_monotonic_increasing
assert df.index.is_unique
assert (df["high"] >= df["low"]).all()
assert (df[["open", "high", "low", "close"]] > 0).all().all()

display(df.tail())
print(f"Loaded {len(df)} candles")


,open,high,low,close,volume
timestamp,,,,,
2026-10-13 15:00:00+00:00,83424.0,83641.6,83138.0,83338.2,129.153123
2026-10-13 16:00:00+00:00,83338.2,83523.3,83300.1,83300.1,100.855714
2026-10-13 17:00:00+00:00,83300.1,83300.1,82875.2,82999.1,131.282241
2026-10-13 18:00:00+00:00,82999.1,83258.0,82883.0,83057.1,99.572201
2026-10-13 19:00:00+00:00,83060.6,83192.2,83060.6,83173.5,27.210602


Loaded 500 candles


In [ ]:

import numpy as np

data = df.copy()

# MACD: 12-period EMA minus 26-period EMA
ema12 = data["close"].ewm(
    span=12, adjust=False
).mean()

ema26 = data["close"].ewm(
    span=26, adjust=False
).mean()

data["macd"] = ema12 - ema26
data["macd_signal"] = data["macd"].ewm(
    span=9, adjust=False
).mean()

# Signal: +1 on bullish crossover, -1 on bearish crossover
previous_macd = data["macd"].shift(1)
previous_signal = data["macd_signal"].shift(1)

data["macd_position"] = np.select(
    [
        (data["macd"] > data["macd_signal"])
        & (previous_macd <= previous_signal),
        (data["macd"] < data["macd_signal"])
        & (previous_macd >= previous_signal),
    ],
    [1, -1],
    default=0,
)

# Bollinger Bands: 20-period mean +/- 2 standard deviations
data["bb_mid"] = data["close"].rolling(20).mean()
std = data["close"].rolling(20).std(ddof=0)

data["bb_upper"] = data["bb_mid"] + 2 * std
data["bb_lower"] = data["bb_mid"] - 2 * std

# Simple mean-reversion entries:
# +1 when price crosses below the lower band;
# -1 when price crosses above the upper band.
previous_close = data["close"].shift(1)
previous_lower = data["bb_lower"].shift(1)
previous_upper = data["bb_upper"].shift(1)

data["bb_signal"] = np.select(
    [
        (data["close"] < data["bb_lower"])
        & (previous_close >= previous_lower),
        (data["close"] > data["bb_upper"])
        & (previous_close <= previous_upper),
    ],
    [1, -1],
    default=0,
)

display(
    data[
        ["close", "macd", "macd_signal",
         "macd_position", "bb_lower",
         "bb_upper", "bb_signal"]
    ].tail(15)
)


,close,macd,macd_signal,macd_position,bb_lower,bb_upper,bb_signal
timestamp,,,,,,,
2026-10-13 05:00:00+00:00,83573.7,-54.635055,-204.143541,0,81551.793815,83773.836185,0
2026-10-13 06:00:00+00:00,83666.8,17.813297,-159.752173,0,81526.021812,83824.188188,0
2026-10-13 07:00:00+00:00,83769.8,82.588350,-111.284069,0,81482.921417,83912.718583,0
2026-10-13 08:00:00+00:00,83487.5,109.877200,-67.051815,0,81476.073880,84001.086120,0
2026-10-13 09:00:00+00:00,83523.5,132.876986,-27.066055,0,81463.384736,84077.875264,0
2026-10-13 10:00:00+00:00,83234.2,126.304431,3.608042,0,81491.446290,84123.333710,0
2026-10-13 11:00:00+00:00,83253.5,121.255228,27.137479,0,81510.519938,84165.500062,0
2026-10-13 12:00:00+00:00,83246.4,115.351091,44.780202,0,81557.238763,84203.261237,0
2026-10-13 13:00:00+00:00,83169.1,103.244419,56.473045,0,81607.840067,84228.569933,0


In [ ]:

def backtest(data, signal_col, fee_rate=0.001):
    result = data.copy()

    # Convert event signals into held positions.
    # MACD crossovers update the position; Bollinger
    # entries use a simple one-candle position for now.
    if signal_col == "macd_position":
        position = result[signal_col].replace(0, np.nan).ffill()
        position = position.fillna(0)
    else:
        position = result[signal_col]

    result["position"] = position.shift(1).fillna(0)

    # Close-to-close returns, earned by prior-candle position.
    result["market_return"] = result["close"].pct_change().fillna(0)

    # Approximate turnover costs on changes in exposure.
    turnover = result["position"].diff().abs()
    turnover.iloc[0] = abs(result["position"].iloc[0])

    result["strategy_return"] = (
        result["position"] * result["market_return"]
        - turnover.fillna(0) * fee_rate
    )

    result["equity"] = (
        1 + result["strategy_return"]
    ).cumprod()

    result["buy_hold"] = (
        1 + result["market_return"]
    ).cumprod()

    peak = result["equity"].cummax()
    drawdown = result["equity"] / peak - 1

    metrics = {
        "total_return_pct": (result["equity"].iloc[-1] - 1) * 100,
        "buy_hold_return_pct": (result["buy_hold"].iloc[-1] - 1) * 100,
        "max_drawdown_pct": drawdown.min() * 100,
        "number_of_position_changes": int(turnover.gt(0).sum()),
    }

    return result, metrics


macd_results, macd_metrics = backtest(
    data, "macd_position"
)

bb_results, bb_metrics = backtest(
    data, "bb_signal"
)

print("MACD:", macd_metrics)
print("Bollinger:", bb_metrics)


MACD: {'total_return_pct': np.float64(-16.807266932595542), 'buy_hold_return_pct': np.float64(-3.790163342783781), 'max_drawdown_pct': -17.862027353505063, 'number_of_position_changes': 45}
Bollinger: {'total_return_pct': np.float64(-4.387004012953144), 'buy_hold_return_pct': np.float64(-3.790163342783781), 'max_drawdown_pct': -5.603587213022587, 'number_of_position_changes': 58}


In [ ]:

from google.colab import drive

drive.mount("/content/drive")



MessageError: Error: credential propagation was unsuccessful

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:

from pathlib import Path

DRIVE_PROJECT = Path(
    "/content/drive/MyDrive/quant-ai-trading"
)

DATA_FOLDER = DRIVE_PROJECT / "data"
RESULTS_FOLDER = DRIVE_PROJECT / "results"

DATA_FOLDER.mkdir(parents=True, exist_ok=True)
RESULTS_FOLDER.mkdir(parents=True, exist_ok=True)

print("Project storage created!")
print("Location:", DRIVE_PROJECT)


Project storage created!
Location: /content/drive/MyDrive/quant-ai-trading


In [ ]:

import json

# Save historical candles
df.to_csv(DATA_FOLDER / "btc_usd_1h.csv")

# Save backtest results
macd_results.to_csv(
    RESULTS_FOLDER / "btc_macd_backtest.csv"
)

bb_results.to_csv(
    RESULTS_FOLDER / "btc_bollinger_backtest.csv"
)

# Save the performance metrics
metrics = {
    "MACD": macd_metrics,
    "Bollinger Bands": bb_metrics,
}

with open(
    RESULTS_FOLDER / "backtest_metrics.json",
    "w"
) as f:
    json.dump(metrics, f, indent=2)

print("Saved files:")
for path in sorted(DRIVE_PROJECT.rglob("*")):
    if path.is_file():
        print(path.relative_to(DRIVE_PROJECT))


Saved files:
data/btc_usd_1h.csv
results/backtest_metrics.json
results/btc_bollinger_backtest.csv
results/btc_macd_backtest.csv
